<div style="border-left: 6px solid #7B5EA7; padding: 14px 18px; background: #f5f2fb; border-radius: 8px;">
<h1 style="color:#2A2440; margin:0;">Where does the planet feel it first?</h1>
<h2 style="color:#7B5EA7; margin:4px 0 12px 0;" dir="rtl">أين يشعر الكوكب أولاً؟</h2>
<div style="color:#2A2440;">
<p><b>Your client:</b> the Arctic Council.</p>
<p><b>Your mission:</b> the Arctic is said to warm faster than anywhere on Earth. The data is here — but it will fight you. Find the trend the client needs, and beware: your first plot will seem to show nothing.</p>
<p><b>Your deliverable:</b> one figure that reveals the trend, and one number — how much September ice the Arctic loses each decade.</p>
<p style="color:#5b5570;"><b>Today's roles:</b> Question Owner · Code Lead · Storyteller — the Storyteller did not write the code.</p>
</div></div>

In [ ]:
# ── The Daily Ritual ────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base = "https://raw.githubusercontent.com/Saleh314Astro/data-science-journey-data/main/clean/"
ice = pd.read_csv(base + "nsidc_seaice_monthly.csv", parse_dates=["date"]).rename(columns={"value": "extent"}).dropna()
ice["year"]  = ice["date"].dt.year
ice["month"] = ice["date"].dt.month
ice.head()

<div style="background:#f5f2fb; border: 1px dashed #7B5EA7; padding: 10px 16px; border-radius: 6px;">

### Prediction 1 — the melting Arctic

If the Arctic is genuinely melting, what should a plot of ALL the monthly data look like? Sketch it.

- **Name:** …
- **My prediction:** …

</div>

In [ ]:
# The naive plot: every month of Arctic sea ice since 1978
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ice["date"], ice["extent"], linewidth=0.8, color="#2f9e94")
ax.set_xlabel("Year")
ax.set_ylabel("Sea ice extent (million km2)")
ax.set_title("Arctic sea ice, every month")
plt.show()

<div style="border-left: 4px solid #2f9e94; padding: 10px 16px; margin-top: 10px;">

### Investigate 1 — why does it show nothing?

1. Describe the plot. Can you state, from this alone, whether the ice is declining?
2. What is the giant up-down repeating every year — and roughly how big is it compared with any long-term change?
3. Propose a way to see the trend **despite** this cycle.

**Our answers:**

- 1 …
- 2 …
- 3 …

</div>

<div dir="rtl" style="background:#f5f2fb; border-right: 4px solid #7B5EA7; padding: 8px 14px; border-radius: 6px; font-size: 0.95em; color:#2A2440;">
<b>الموسمية</b> <span dir="ltr">(seasonality)</span>: سلسلة زمنية قد تكون مجموع مكوّنات — دورة موسمية + اتجاه طويل + ضجيج. هنا الدورة الموسمية أضخم من الاتجاه بعشرات المرّات، فتُخفيه تماماً في الرسم الساذج.
</div>

<div dir="rtl" style="background:#f5f2fb; border-right: 4px solid #7B5EA7; padding: 8px 14px; border-radius: 6px; font-size: 0.95em; color:#2A2440;">
<b>التجميع والمناخ المتوسط</b> <span dir="ltr">(groupby / climatology)</span>: <span dir="ltr">ice.groupby("month")["extent"].mean()</span> يقسم البيانات إلى 12 مجموعة (شهر لكل مجموعة) ويعطي متوسط كل شهر. هذا يكشف شكل الدورة الموسمية نفسها.
</div>

In [ ]:
# Diagnose the problem: average each calendar month across all years
clim = ice.groupby("month")["extent"].mean()      # the seasonal cycle itself
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(clim.index, clim.values, marker="o", color="#7B5EA7")
ax.set_xlabel("Month (1 = Jan)")
ax.set_ylabel("Mean extent (million km2)")
ax.set_title("This seasonal swing is what was hiding the trend")
plt.show()
print("summer low (Sep) vs winter high (Mar):", round(clim.min(),1), "vs", round(clim.max(),1), "million km2")

<div style="background:#f5f2fb; border: 1px dashed #7B5EA7; padding: 10px 16px; border-radius: 6px;">

### Prediction 2 — September, alone

September is the yearly LOW — the ice that survives the summer melt. If you plot ONLY September across the years, what will you finally see?

- **Name:** …
- **My prediction:** …

</div>

In [ ]:
# YOUR CODE HERE — escape the trap: isolate ONE month and plot it over the years
# 1) sep = the rows where ice["month"] == 9
# 2) plot sep["year"] vs sep["extent"]  (add axis labels with units)
#
# sep = ice[ ... ]
# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot( ... )
# ax.set_xlabel("Year"); ax.set_ylabel("September extent (million km2)")
# plt.show()

<div style="border-left: 4px solid #2f9e94; padding: 10px 16px; margin-top: 10px;">

### Investigate 2 — the honest witness

1. Compare this to your first plot. What made the trend appear?
2. Read the decline: roughly how much September ice is lost, and is it steady or speeding up?
3. Why is September the right month to watch, not March?

**Our answers:**

- 1 …
- 2 …
- 3 …

</div>

In [ ]:
# How fast is September ice disappearing?
sep = ice[ice["month"] == 9]
slope = np.polyfit(sep["year"], sep["extent"], 1)[0]
print("September trend:", round(slope * 10, 2), "million km2 per decade")
print("lowest September on record:", int(sep.loc[sep["extent"].idxmin(), "year"]),
      "->", round(sep["extent"].min(), 2), "million km2")

<div dir="rtl" style="background:#f5f2fb; border-right: 4px solid #7B5EA7; padding: 8px 14px; border-radius: 6px; font-size: 0.95em; color:#2A2440;">
<b>تغذية البياض الراجعة</b> <span dir="ltr">(albedo feedback)</span>: الجليد أبيض يعكس الشمس <span dir="ltr">(α≈0.6)</span>؛ الماء داكن يبتلعها <span dir="ltr">(α≈0.06)</span>. يذوب الجليد فينكشف ماء داكن يمتصّ حرارة أكثر فيذوب جليد أكثر — مضخّم، لا انفلات. لاحظوا: هذه التغذية تهاجم <span dir="ltr">α</span>، أحد مدخلات معادلة الأمس.
</div>

<div style="background:#f5f2fb; border: 1px dashed #7B5EA7; padding: 10px 16px; border-radius: 6px;">

### Stretch lane — find a volcano in the temperature record

In June 1991 Mount Pinatubo threw a veil of dust into the stratosphere and cooled the whole planet for about two years. Remove the long trend from the temperature record and hunt the dip.

</div>

In [ ]:
# Stretch — hunt a volcano in the temperature residuals
t = pd.read_csv(base + "gistemp_monthly.csv", parse_dates=["date"]).rename(columns={"value": "anomaly"}).dropna()
t["resid"]  = t["anomaly"] - t["anomaly"].rolling(120, center=True).mean()   # remove the long trend
t["smooth"] = t["resid"].rolling(12, center=True).mean()                     # calm the noise

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t["date"], t["smooth"], color="#2f9e94")
ax.axhline(0, color="gray")
ax.set_xlim(pd.Timestamp("1988-01-01"), pd.Timestamp("2000-01-01"))
ax.set_xlabel("Year")
ax.set_ylabel("Temperature residual (degC)")
ax.set_title("What remains after the trend is removed")
plt.show()

window = t[(t["date"] >= "1991-06-01") & (t["date"] <= "1994-12-01")]
print("coolest month after the June 1991 eruption:", t.loc[window["smooth"].idxmin(), "date"].strftime("%Y-%m"))

<div style="border: 2px solid #7B5EA7; padding: 12px 18px; background:#f5f2fb; border-radius: 8px;">

## Team claim — the sentence for the Arctic Council

One sentence: the September number (with units), how confident you are, and one line on why your first plot hid it.

Structure, not content — Omar once wrote: "Arctic September ice is falling by about X million km2 per decade; the trend was invisible at first because a seasonal cycle Y times larger sat on top of it."

**Our team's sentence:** …

**Today's Question Owner:** …

</div>

<div style="border-left: 6px solid #2A2440; padding: 12px 18px;">

### Today's reflection — named, and it will be read

In one line: your first plot was true and yet it hid the answer. What does that teach you about a graph that "shows nothing"?

- **Name:** …
- **My line:** …

<p style="margin-top:14px; color:#7B5EA7;">The ice keeps a slower calendar than the seasons — and its September is quietly getting shorter. تَفَكَّروا.</p>

</div>

<p style="text-align:center; margin-top:10px;">
<span style="display:inline-block;width:14px;height:14px;border-radius:50%;background:#2f9e94;"></span>
<span style="display:inline-block;width:14px;height:14px;border-radius:50%;background:#2f9e94;"></span>
<span style="display:inline-block;width:14px;height:14px;border-radius:50%;background:#2f9e94;"></span>
<span style="display:inline-block;width:14px;height:14px;border-radius:50%;background:#2f9e94;"></span>
<span style="display:inline-block;width:14px;height:14px;border-radius:50%;background:#e5e2ec;"></span>
<br><span style="font-size:11px;color:#5b5570;">Cryosphere · الغلاف الجليدي</span>
</p>